# CVaR Sensitivity On Validation

This notebook runs the gamma grid on validation selected weeks only.

CVaR parameters:
- `alpha`: tail confidence (default 0.95)
- `gamma`: risk-aversion weight

## Selection Rule And Leakage Guardrail

Gamma must be selected on validation only.

Suggested rule:
Choose the smallest gamma that materially improves downside risk while retaining most adjusted profit.

Test-week gamma sweeps are ex-post sensitivity analysis only and are not parameter selection.

In [ ]:
from pathlib import Path
import sys
from dataclasses import replace

repo_root = Path.cwd()
while repo_root.name != "Thesis" and repo_root.parent != repo_root:
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "scripts" / "Data" / "03_Hydrogen_Test_Case"))

from hydrogen.plant_parameters import load_hydrogen_config
from hydrogen.rolling_horizon import run_cvar_gamma_validation_sweep

config = load_hydrogen_config(repo_root / "scripts" / "Data" / "03_Hydrogen_Test_Case" / "configs" / "base_hydrogen.yaml")
config = replace(config, experiment=replace(config.experiment, dataset_split="validation"))
sweep = run_cvar_gamma_validation_sweep(config_or_path=config)
sweep

In [ ]:
selection = sweep.sort_values(["realised_empirical_CVaR", "adjusted_net_profit_eur"], ascending=[True, False]).head(1)
selection

In [ ]:
if not selection.empty:
    selected_gamma = float(selection["gamma"].iloc[0])
    out_dir = Path(selection["run_dir"].iloc[0]).parent
    sweep.to_csv(out_dir / "cvar_gamma_selection_summary.csv", index=False)
    (out_dir / "selected_cvar_config.json").write_text(
        '{"selection_split": "validation", "selected_gamma": ' + str(selected_gamma) + '}',
        encoding="utf-8",
    )
    print("Saved gamma selection artifacts to", out_dir)